# Design simulation

## How to read the statistics

The primary metric is **RMSE**: the root mean squared error of the posterior mean of ability versus the simulated true ability. Lower is better. The decision threshold was 0.45. Monte Carlo intervals are the standard error of that RMSE across 20 simulated experiments, not participant-level confidence intervals.

Other diagnostics: Pearson *r* (higher better), MAE, bias (near 0 better), mean posterior SD, and 95% interval coverage (near 0.95 better).


## Simulation assumptions

Responses are generated from a 1PL model, or a 3PL with guessing 0.25. True abilities are Normal(0, 1). The item bank is the same 32-item judgement-calibrated set used in the demo. Each replicate has 40 participants. Stopping is disabled on the matched-budget grid. A separate stopping-enabled cell uses min 8 / max 16 / SE 0.40. Costs use $12/hour, 35 s fixed and 8 s per scored item. Attrition, recruiter fees, and bonuses are omitted.


In [1]:

from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "plotly_mimetype"
pio.templates.default = "plotly_white"

here = Path.cwd()
root = here if (here / "experiment.py").exists() else here.parents[2]
results = pd.read_csv(root / "audit/simulate/design/results.csv")
results.head()


,result_id,scenario_id,analysis_id,parameter_id,method,policy,max_items,stop_early,guessing,n_participants,metric,estimate,mc_se,decision_metric,decision_value,decision_threshold,meets_requirement,participant_payment,currency
0,max_information_m8_g0.00_fixed__rmse,max_information_m8_g0.00_fixed,ability,rmse,precision-estimation,max_information,8,False,0.0,40,rmse,0.623675,0.014374,rmse,0.623675,0.45,False,13.2,USD
1,max_information_m8_g0.00_fixed__mae,max_information_m8_g0.00_fixed,ability,mae,precision-estimation,max_information,8,False,0.0,40,mae,0.501915,0.011261,rmse,NaN,0.45,False,13.2,USD
2,max_information_m8_g0.00_fixed__bias,max_information_m8_g0.00_fixed,ability,bias,precision-estimation,max_information,8,False,0.0,40,bias,0.007423,0.030857,rmse,NaN,0.45,False,13.2,USD
3,max_information_m8_g0.00_fixed__pearson_r,max_information_m8_g0.00_fixed,ability,pearson_r,precision-estimation,max_information,8,False,0.0,40,pearson_r,0.784528,0.013742,rmse,NaN,0.45,False,13.2,USD
4,max_information_m8_g0.00_fixed__mean_posterior_sd,max_information_m8_g0.00_fixed,ability,mean_posterior_sd,precision-estimation,max_information,8,False,0.0,40,mean_posterior_sd,0.603731,0.000327,rmse,NaN,0.45,False,13.2,USD


## Power analysis

In [2]:

rmse = results[(results.metric == "rmse") & (~results.stop_early)].copy()
rmse["scenario"] = rmse.apply(
    lambda r: ("Matching 1PL" if r.guessing == 0 else "3PL guessing")
    + ", "
    + ("Max information" if r.policy == "max_information" else "Random"),
    axis=1,
)
fig = go.Figure()
for name, group in rmse.groupby("scenario"):
    group = group.sort_values("max_items")
    fig.add_trace(
        go.Scatter(
            x=group.max_items,
            y=group.estimate,
            error_y=dict(type="data", array=group.mc_se, visible=True),
            mode="lines+markers",
            name=name,
        )
    )
fig.add_hline(y=0.45, line_dash="dash", annotation_text="RMSE threshold 0.45")
fig.update_layout(
    height=420,
    margin=dict(l=70, r=30, t=90, b=60),
    title=dict(text="Ability RMSE at matched test length", x=0, xanchor="left"),
    xaxis_title="Scored items",
    yaxis_title="RMSE",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0, title_text=""),
    xaxis=dict(tickvals=[8, 12, 16]),
)
fig


## Adaptive procedure

In [3]:

stop = results[(results.metric == "rmse")].copy()
table = stop.pivot_table(
    index=["policy", "guessing", "stop_early", "max_items"],
    values=["estimate", "mc_se", "meets_requirement", "participant_payment"],
    aggfunc="first",
).reset_index()
table


,policy,guessing,stop_early,max_items,estimate,mc_se,meets_requirement,participant_payment
0,max_information,0.00,False,8,0.623675,0.014374,False,13.200000
1,max_information,0.00,False,12,0.523672,0.013250,False,17.466667
2,max_information,0.00,False,16,0.477012,0.013667,False,21.733333
3,max_information,0.00,True,16,0.484093,0.011040,False,21.733333
4,max_information,0.25,False,8,0.815297,0.024027,False,13.200000
5,max_information,0.25,False,12,0.826882,0.027550,False,17.466667
6,max_information,0.25,False,16,0.788263,0.026925,False,21.733333
7,max_information,0.25,True,16,0.793226,0.021704,False,21.733333
8,random,0.00,False,8,0.639548,0.019323,False,13.200000
9,random,0.00,False,12,0.559556,0.012408,False,17.466667


## Decision

No candidate design met RMSE ≤ 0.45. At 16 well-specified items, max-information RMSE was about 0.48 versus 0.53 for random order: a modest gain, not enough for the pre-set criterion. The SE-0.40 stopping rule never shortened the test in these simulations (mean length 16). The 3PL-guessing scenario is substantially worse for both policies.

The demo still uses max-information CAT because the goal is to exercise the adaptive workflow, not to claim a powered arithmetic study. For a real study, enlarge or empirically calibrate the item bank, or relax the precision target.
